# 04 — Incertidumbre y política de abastecimiento

**Caso Tostao | Decisión SKU–Tienda | Horizonte: 7 días**

## Objetivo
Transformar el pronóstico ETS semanal y sus errores fuera de muestra en una recomendación de pedido prudente, evitando que el modelo acumule inventario excesivo.

La política propuesta:
- Demanda base: pronóstico de los próximos 7 días.
- Protección: cuantiles empíricos Q85, Q90 y Q95 de la demanda.
- Política inicial: Q90.
- Tope de inventario objetivo: cobertura equivalente a 8 días (7 días + 1 día adicional).
- Pedido: inventario objetivo menos stock actual disponible.

El día adicional es un **supuesto inicial de política**, no un dato de capacidad ni una regla operacional confirmada. El análisis no considera pedidos en tránsito porque no se suministró esa información.


## 1. Fundamento de la política

Para cada SKU–Tienda se estima la distribución de demanda acumulada de los próximos siete días mediante remuestreo de errores semanales fuera de muestra.

Se calculan tres escenarios:
- **Q85:** menor protección.
- **Q90:** política inicial recomendada.
- **Q95:** mayor protección.

Para evitar que los cuantiles eleven el inventario sin límite, se aplica un máximo equivalente a ocho días de demanda promedio:

\[
S_{max}=\hat D_{7d}\times\frac{8}{7}
\]

Para cada nivel de servicio:

\[
S_{\alpha}=\min(Q_{\alpha}(D), S_{max})
\]

Y el pedido sugerido:

\[
Pedido_{\alpha}=\max(0,\lceil S_{\alpha}-StockActual\rceil)
\]

**Importante:** Q90 es un cuantil empírico aproximado construido con los errores disponibles; no garantiza un 90% real de nivel de servicio. El límite de ocho días controla la cobertura objetivo, pero no equivale a una restricción física de capacidad.


## 2. Configuración y carga de archivos

Carga los cuatro CSV originales y los dos archivos exportados por el Notebook 03 final:
- `forecast_ets_semanal_sku_tienda.csv`
- `backtesting_modelos_base.csv`

Si no tienes los dos archivos de salida, ejecuta primero el Notebook 03 final.

In [ ]:
import os, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)

from google.colab import files
required_files = [
    "inventario_actual.csv", "catalogo_productos.csv",
    "forecast_ets_semanal_sku_tienda.csv",
    "backtesting_modelos_base.csv"
]
missing = [f for f in required_files if not Path("/content", f).exists()]
if missing:
    print("Carga los siguientes archivos:", missing)
    files.upload()

def locate(name):
    for p in [Path("/content")/name, Path(name)]:
        if p.exists():
            return p
    raise FileNotFoundError(f"No se encontró {name}")

inventario = pd.read_csv(locate("inventario_actual.csv"))
productos = pd.read_csv(locate("catalogo_productos.csv"))
forecast_weekly = pd.read_csv(locate("forecast_ets_semanal_sku_tienda.csv"))
backtest = pd.read_csv(locate("backtesting_modelos_base.csv"))

print("Inventario:", inventario.shape)
print("Catálogo:", productos.shape)
print("Forecast:", forecast_weekly.shape)
print("Backtesting:", backtest.shape)

Inventario: (160, 3)
Catálogo: (8, 6)
Forecast: (160, 3)
Backtesting: (13440, 7)


## 3. Validación de esquemas y llaves

Se detiene la ejecución si faltan columnas necesarias o existen duplicados en las llaves de inventario/catálogo. Es preferible detectar una incompatibilidad a producir recomendaciones incorrectas.

In [ ]:
keys = ["id_tienda", "id_producto"]
required_inventory = set(keys + ["stock_actual"])
required_catalog = set(["id_producto","costo_unitario","precio_venta","costo_almacenamiento_semanal"])
required_forecast = set(keys + ["demanda_pronosticada_7d"])
required_backtest = set(keys + ["fecha","y","pred","modelo","fold"])

for label, df, cols in [
    ("inventario", inventario, required_inventory),
    ("catálogo", productos, required_catalog),
    ("forecast", forecast_weekly, required_forecast),
    ("backtesting", backtest, required_backtest)
]:
    missing_cols = cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"Faltan columnas en {label}: {sorted(missing_cols)}")

for label, df, subset in [
    ("inventario", inventario, keys),
    ("catálogo", productos, ["id_producto"]),
    ("forecast", forecast_weekly, keys)
]:
    if df.duplicated(subset).any():
        raise ValueError(f"Hay duplicados en la llave de {label}: {subset}")

forecast_weekly["demanda_pronosticada_7d"] = pd.to_numeric(
    forecast_weekly["demanda_pronosticada_7d"], errors="coerce"
).clip(lower=0)
inventario["stock_actual"] = pd.to_numeric(inventario["stock_actual"], errors="coerce")
for c in ["costo_unitario","precio_venta","costo_almacenamiento_semanal"]:
    productos[c] = pd.to_numeric(productos[c], errors="coerce")

print("Modelos disponibles en backtesting:", backtest["modelo"].unique())
print("Filas ETS:", (backtest["modelo"]=="ETS_Aditivo_7").sum())

Modelos disponibles en backtesting: ['Naive' 'Seasonal_Naive_7' 'Moving_Average_7' 'ETS_Aditivo_7']
Filas ETS: 3360


## 4. Construcción de errores semanales fuera de muestra

El archivo de backtesting contiene predicciones diarias de los tres folds. Para aproximar el error relevante a la decisión semanal, se suman demanda real y pronosticada dentro de cada combinación SKU–Tienda y fold.

Se calcula el error relativo:

\[
r = \frac{D_{real}-D_{pronosticada}}{D_{pronosticada}}
\]

Estos residuos se agrupan por producto para compartir información entre tiendas. Si un producto tiene pocos errores válidos, se utiliza la distribución global. El procedimiento genera una distribución empírica aproximada, no un intervalo estadístico paramétrico.

In [ ]:
bt_ets = backtest[backtest["modelo"]=="ETS_Aditivo_7"].copy()
if bt_ets.empty:
    raise ValueError("No hay predicciones ETS_Aditivo_7 en backtesting_modelos_base.csv")

bt_ets["fecha"] = pd.to_datetime(bt_ets["fecha"])
weekly_errors = (bt_ets.groupby(keys+["fold"], as_index=False)
                 .agg(demanda_real=("y","sum"),
                      demanda_predicha=("pred","sum"),
                      dias_evaluados=("fecha","nunique")))
weekly_errors["error"] = weekly_errors["demanda_real"] - weekly_errors["demanda_predicha"]
weekly_errors["error_relativo"] = (
    weekly_errors["error"] / weekly_errors["demanda_predicha"].clip(lower=1)
)
weekly_errors = weekly_errors.replace([np.inf,-np.inf], np.nan).dropna(subset=["error_relativo"])
weekly_errors = weekly_errors.merge(productos[["id_producto"]], on="id_producto", how="left")

print("Errores semanales:", len(weekly_errors))
print("Series con errores:", weekly_errors[keys].drop_duplicates().shape[0])
display(weekly_errors.head())

Errores semanales: 480
Series con errores: 160


,id_tienda,id_producto,fold,demanda_real,demanda_predicha,dias_evaluados,error,error_relativo
0,STORE_01,PROD_001,1,39,51.179380,7,-12.179380,-0.237974
1,STORE_01,PROD_001,2,45,49.577938,7,-4.577938,-0.092338
2,STORE_01,PROD_001,3,51,49.069451,7,1.930549,0.039343
3,STORE_01,PROD_002,1,61,71.000000,7,-10.000000,-0.140845
4,STORE_01,PROD_002,2,49,61.000000,7,-12.000000,-0.196721


## 5. Diagnóstico de incertidumbre y cobertura

Se revisa la dispersión de los errores relativos. Con solo tres folds, cada serie tiene pocas observaciones; por eso no se estiman intervalos independientes por SKU–Tienda. Se utiliza pooling por producto y respaldo global para reducir la inestabilidad.

Los intervalos resultantes deben describirse como **intervalos predictivos empíricos aproximados**, no como intervalos de confianza garantizados.

In [ ]:
uncertainty_summary = (weekly_errors.groupby("id_producto")["error_relativo"]
                       .agg(n="count", mediana="median", q10=lambda s:s.quantile(.10),
                            q90=lambda s:s.quantile(.90), desviacion="std")
                       .reset_index())
display(uncertainty_summary)

print("Cuantiles globales del error relativo:")
display(weekly_errors["error_relativo"].quantile([.05,.10,.50,.90,.95,.99]).to_frame("error_relativo"))

,id_producto,n,mediana,q10,q90,desviacion
0,PROD_001,60,-0.029869,-0.219097,0.210112,0.190849
1,PROD_002,60,0.025482,-0.237918,0.368089,0.237779
2,PROD_003,60,0.007463,-0.183961,0.236411,0.187439
3,PROD_004,60,-0.022246,-0.187737,0.366667,0.214533
4,PROD_005,60,-0.002959,-0.169935,0.153623,0.156795
5,PROD_006,60,-0.033306,-0.197017,0.238525,0.215057
6,PROD_007,60,0.046745,-0.212162,0.217505,0.169224
7,PROD_008,60,-0.004167,-0.247438,0.270543,0.203837


Cuantiles globales del error relativo:


,error_relativo
0.05,-0.260459
0.10,-0.214286
0.50,0.000000
0.90,0.270543
0.95,0.394435
0.99,0.564331


## 6. Generación de escenarios de demanda

Para cada SKU–Tienda se toma el forecast puntual de siete días y se le aplica un remuestreo de errores relativos del producto correspondiente. Si hay menos de 30 observaciones para el producto, se usa el conjunto global.

\[
D^{(b)} = \max(0, \hat D \times (1+r^{(b)}))
\]

La simulación permite obtener cuantiles de demanda y estimar el costo esperado para diferentes niveles de inventario. Se utiliza una simulación de 10.000 escenarios por combinación.

In [ ]:
N_SIM = 10000
MIN_PRODUCT_ERRORS = 30

global_residuals = weekly_errors["error_relativo"].dropna().to_numpy()
if len(global_residuals) < 10:
    raise ValueError("Hay muy pocos errores fuera de muestra para estimar incertidumbre.")

product_residuals = {
    product: g["error_relativo"].dropna().to_numpy()
    for product, g in weekly_errors.groupby("id_producto")
}

def get_residual_pool(product):
    arr = product_residuals.get(product, np.array([]))
    if len(arr) >= MIN_PRODUCT_ERRORS:
        return arr, "producto"
    return global_residuals, "global"

# Un conjunto reproducible de escenarios para cada producto.
scenario_rows = []
scenario_cache = {}
for _, row in forecast_weekly.iterrows():
    product = row["id_producto"]
    point = max(0.0, float(row["demanda_pronosticada_7d"]))
    pool, source = get_residual_pool(product)
    sampled_residuals = rng.choice(pool, size=N_SIM, replace=True)
    demand_scenarios = np.maximum(0, point * (1 + sampled_residuals))
    scenario_cache[(row["id_tienda"], product)] = demand_scenarios
    scenario_rows.append({
        "id_tienda":row["id_tienda"],
        "id_producto":product,
        "demanda_puntual":point,
        "q05":np.quantile(demand_scenarios,.05),
        "q10":np.quantile(demand_scenarios,.10),
        "q50":np.quantile(demand_scenarios,.50),
        "q90":np.quantile(demand_scenarios,.90),
        "q95":np.quantile(demand_scenarios,.95),
        "q99":np.quantile(demand_scenarios,.99),
        "fuente_residuos":source,
        "n_residuos_pool":len(pool)
    })

demand_distribution = pd.DataFrame(scenario_rows)
display(demand_distribution.head(10))

,id_tienda,id_producto,demanda_puntual,q05,q10,q50,q90,q95,q99,fuente_residuos,n_residuos_pool
0,STORE_01,PROD_001,49.270432,37.473286,38.321447,47.375415,59.445847,65.693909,81.447040,producto,60
1,STORE_01,PROD_002,69.000000,47.533333,52.192308,70.780645,97.163265,100.363636,110.687500,producto,60
2,STORE_01,PROD_003,68.000000,47.304348,54.777778,68.000000,83.508772,90.666667,101.244444,producto,60
3,STORE_01,PROD_004,46.000000,31.625000,37.375000,44.805195,64.121212,65.863636,68.566038,producto,60
4,STORE_01,PROD_005,73.000000,56.940000,60.603774,72.568047,85.166667,103.323077,115.000000,producto,60
5,STORE_01,PROD_006,80.000000,59.166667,63.255814,77.335491,99.672131,109.010989,142.222222,producto,60
6,STORE_01,PROD_007,83.000000,64.555556,65.427928,86.807339,101.009434,104.378788,129.382353,producto,60
7,STORE_01,PROD_008,70.000000,47.638889,51.711712,70.000000,88.846154,92.222222,104.027778,producto,60
8,STORE_02,PROD_001,71.000000,54.103821,55.468750,69.489362,85.663043,94.666667,117.367347,producto,60
9,STORE_02,PROD_002,85.000000,55.135135,64.294872,87.138365,119.693878,123.636364,136.354167,producto,60


## 7. Política de inventario: cuantiles con límite de cobertura

Se abandona como política principal el fractil económico Newsvendor basado únicamente en margen y costo de almacenamiento. El costo de almacenamiento disponible no representa necesariamente el costo completo de sobreinventario y podía empujar el inventario objetivo hacia cuantiles extremos.

La decisión se basa ahora en niveles de protección Q85, Q90 y Q95, con un máximo de ocho días de cobertura. Se mantiene el costo esperado como indicador comparativo, no como criterio único de decisión.


In [ ]:
decision = (forecast_weekly
            .merge(inventario[keys+["stock_actual"]], on=keys, how="left", validate="one_to_one")
            .merge(productos[["id_producto","costo_unitario","precio_venta",
                              "costo_almacenamiento_semanal"]],
                   on="id_producto", how="left", validate="many_to_one")
            .merge(demand_distribution, on=keys, how="left", validate="one_to_one"))

if decision["stock_actual"].isna().any():
    raise ValueError("Hay combinaciones sin stock_actual; no se puede calcular pedido.")
if decision[["costo_unitario","precio_venta","costo_almacenamiento_semanal"]].isna().any().any():
    raise ValueError("Hay costos o precios faltantes en el catálogo.")

decision["margen_unitario"] = (
    decision["precio_venta"] - decision["costo_unitario"]
).clip(lower=0)
decision["costo_faltante_unitario"] = decision["margen_unitario"]
decision["costo_exceso_unitario"] = decision["costo_almacenamiento_semanal"].clip(lower=0)

denom = decision["costo_faltante_unitario"] + decision["costo_exceso_unitario"]
decision["fractil_critico"] = np.where(
    denom > 0, decision["costo_faltante_unitario"]/denom, 0.5
).clip(0.01,0.99)

print("Resumen del fractil crítico:")
display(decision["fractil_critico"].describe(percentiles=[.5,.9,.95,.99]).to_frame())

Resumen del fractil crítico:


,fractil_critico
count,160.000000
mean,0.988182
std,0.002284
min,0.983607
50%,0.989129
90%,0.990000
95%,0.990000
99%,0.990000
max,0.990000


## 8. Cálculo de inventario objetivo y pedido sugerido

Para cada combinación SKU–Tienda se calcula el inventario objetivo de Q85, Q90 y Q95. Cada objetivo se limita al máximo de cobertura de ocho días. La política Q90 queda marcada como escenario inicial para la recomendación operativa.


In [ ]:
# Parámetros explícitos de política
HORIZON_DAYS = 7
SAFETY_DAYS = 1
MAX_COVERAGE_DAYS = HORIZON_DAYS + SAFETY_DAYS
SERVICE_LEVELS = [0.85, 0.90, 0.95]
DEFAULT_POLICY = 0.90

policy_rows = []
for _, row in decision.iterrows():
    key = (row["id_tienda"], row["id_producto"])
    samples = scenario_cache[key]
    daily_mean = max(0.0, float(row["demanda_pronosticada_7d"])) / HORIZON_DAYS
    max_inventory = int(np.ceil(daily_mean * MAX_COVERAGE_DAYS))

    for sl in SERVICE_LEVELS:
        raw_quantile = float(np.quantile(samples, sl))
        target = min(int(np.ceil(raw_quantile)), max_inventory)
        order = max(0, int(np.ceil(target - row["stock_actual"])))

        shortage = np.maximum(samples - target, 0)
        excess = np.maximum(target - samples, 0)
        cu = max(0.0, float(row["margen_unitario"]))
        co = max(0.0, float(row["costo_almacenamiento_semanal"]))
        expected_cost = float(np.mean(cu * shortage + co * excess))

        policy_rows.append({
            "id_tienda": row["id_tienda"],
            "id_producto": row["id_producto"],
            "politica": f"Q{int(sl*100)}",
            "nivel_cuantil": sl,
            "demanda_pronosticada_7d": float(row["demanda_pronosticada_7d"]),
            "cuantil_sin_tope": raw_quantile,
            "tope_cobertura_8d": max_inventory,
            "inventario_objetivo": target,
            "stock_actual": float(row["stock_actual"]),
            "pedido_sugerido": order,
            "costo_esperado_estimado": expected_cost
        })

policy_comparison = pd.DataFrame(policy_rows)
display(policy_comparison.head(15))


,id_tienda,id_producto,politica,nivel_cuantil,demanda_pronosticada_7d,cuantil_sin_tope,tope_cobertura_8d,inventario_objetivo,stock_actual,pedido_sugerido,costo_esperado_estimado
0,STORE_01,PROD_001,Q85,0.85,49.270432,57.314584,57,57,1.0,56,2140.407723
1,STORE_01,PROD_001,Q90,0.90,49.270432,59.445847,57,57,1.0,56,2140.407723
2,STORE_01,PROD_001,Q95,0.95,49.270432,65.693909,57,57,1.0,56,2140.407723
3,STORE_01,PROD_002,Q85,0.85,69.000000,88.567164,79,79,10.0,69,8113.998471
4,STORE_01,PROD_002,Q90,0.90,69.000000,97.163265,79,79,10.0,69,8113.998471
5,STORE_01,PROD_002,Q95,0.95,69.000000,100.363636,79,79,10.0,69,8113.998471
6,STORE_01,PROD_003,Q85,0.85,68.000000,80.830189,78,78,33.0,45,5600.627415
7,STORE_01,PROD_003,Q90,0.90,68.000000,83.508772,78,78,33.0,45,5600.627415
8,STORE_01,PROD_003,Q95,0.95,68.000000,90.666667,78,78,33.0,45,5600.627415
9,STORE_01,PROD_004,Q85,0.85,46.000000,61.333333,53,53,11.0,42,3947.626186


## 9. Comparación de escenarios y selección de política

Se comparan Q85, Q90 y Q95 por:
- unidades totales sugeridas;
- inventario objetivo total;
- combinaciones con pedido positivo;
- costo esperado estimado bajo el proxy disponible.

La política Q90 se utiliza como escenario base para el archivo final. Q85 y Q95 quedan disponibles para discusión con abastecimiento.


In [ ]:
policy_summary = (
    policy_comparison.groupby("politica", as_index=False)
    .agg(
        pedido_total_unidades=("pedido_sugerido", "sum"),
        inventario_objetivo_total=("inventario_objetivo", "sum"),
        combinaciones_con_pedido=("pedido_sugerido", lambda s: int((s > 0).sum())),
        costo_esperado_estimado=("costo_esperado_estimado", "sum")
    )
)
display(policy_summary)

recommendations = policy_comparison[
    policy_comparison["politica"] == f"Q{int(DEFAULT_POLICY*100)}"
].copy()
recommendations = recommendations.sort_values(
    ["pedido_sugerido", "costo_esperado_estimado"], ascending=[False, False]
)
display(recommendations.head(25))


,politica,pedido_total_unidades,inventario_objetivo_total,combinaciones_con_pedido,costo_esperado_estimado
0,Q85,13740,17063,154,1.131283e+06
1,Q90,13805,17128,154,1.117018e+06
2,Q95,13805,17128,154,1.117018e+06


,id_tienda,id_producto,politica,nivel_cuantil,demanda_pronosticada_7d,cuantil_sin_tope,tope_cobertura_8d,inventario_objetivo,stock_actual,pedido_sugerido,costo_esperado_estimado
373,STORE_16,PROD_005,Q90,0.9,264.0,308.000000,302,302,25.0,277,10721.760762
187,STORE_08,PROD_007,Q90,0.9,277.0,338.555556,317,317,56.0,261,13942.646670
445,STORE_19,PROD_005,Q90,0.9,198.0,228.130435,227,227,17.0,210,7615.759326
376,STORE_16,PROD_006,Q90,0.9,183.0,226.500000,210,210,3.0,207,15710.845044
67,STORE_03,PROD_007,Q90,0.9,180.0,219.056604,206,206,4.0,202,8958.923002
253,STORE_11,PROD_005,Q90,0.9,178.0,205.086957,204,204,8.0,196,6622.289425
118,STORE_05,PROD_008,Q90,0.9,179.0,227.192308,205,205,12.0,193,18986.915449
379,STORE_16,PROD_007,Q90,0.9,175.0,212.971698,200,200,7.0,193,8971.509843
139,STORE_06,PROD_007,Q90,0.9,199.0,242.179245,228,228,37.0,191,9516.848706
181,STORE_08,PROD_005,Q90,0.9,212.0,247.333333,243,243,52.0,191,8583.795904


## 10. Revisión de recomendaciones

La tabla final contiene una fila por SKU–Tienda para la política Q90. Antes de usarla como orden real, validar inventario disponible, pedidos en tránsito, lead time, múltiplos de compra, perecibilidad y restricciones operativas.


In [ ]:
final_columns = [
    "id_tienda", "id_producto", "politica",
    "demanda_pronosticada_7d", "cuantil_sin_tope", "tope_cobertura_8d",
    "stock_actual", "inventario_objetivo", "pedido_sugerido",
    "costo_esperado_estimado"
]
recommendations = recommendations[final_columns].copy()
display(recommendations)
print("Combinaciones recomendadas:", recommendations[keys].drop_duplicates().shape[0])


,id_tienda,id_producto,politica,demanda_pronosticada_7d,cuantil_sin_tope,tope_cobertura_8d,stock_actual,inventario_objetivo,pedido_sugerido,costo_esperado_estimado
373,STORE_16,PROD_005,Q90,264.0,308.000000,302,25.0,302,277,10721.760762
187,STORE_08,PROD_007,Q90,277.0,338.555556,317,56.0,317,261,13942.646670
445,STORE_19,PROD_005,Q90,198.0,228.130435,227,17.0,227,210,7615.759326
376,STORE_16,PROD_006,Q90,183.0,226.500000,210,3.0,210,207,15710.845044
67,STORE_03,PROD_007,Q90,180.0,219.056604,206,4.0,206,202,8958.923002
...,...,...,...,...,...,...,...,...,...,...
466,STORE_20,PROD_004,Q90,36.0,49.090909,42,42.0,42,0,2905.471925
154,STORE_07,PROD_004,Q90,33.0,46.000000,38,55.0,38,0,2843.784847
193,STORE_09,PROD_001,Q90,51.0,63.363636,59,75.0,59,0,2328.322183
394,STORE_17,PROD_004,Q90,18.0,24.545455,21,25.0,21,0,1431.382640


Combinaciones recomendadas: 160


## 11. Sensibilidad al límite de cobertura

El día adicional es un supuesto. Se compara el efecto de topes de 7, 8 y 9 días, manteniendo Q90. Esto permite mostrar cuánto cambia el pedido si abastecimiento prefiere operar sin protección adicional, con un día o con dos días adicionales.

No se debe interpretar este análisis como capacidad física de tienda: para ello se requeriría capacidad real por producto o familia.


In [ ]:
coverage_sensitivity_rows = []
for extra_days in [0, 1, 2]:
    max_days = HORIZON_DAYS + extra_days
    total_order = 0
    total_target = 0
    for _, row in decision.iterrows():
        samples = scenario_cache[(row["id_tienda"], row["id_producto"])]
        q90 = float(np.quantile(samples, DEFAULT_POLICY))
        daily_mean = max(0.0, float(row["demanda_pronosticada_7d"])) / HORIZON_DAYS
        cap = int(np.ceil(daily_mean * max_days))
        target = min(int(np.ceil(q90)), cap)
        order = max(0, int(np.ceil(target - row["stock_actual"])))
        total_order += order
        total_target += target
    coverage_sensitivity_rows.append({
        "cobertura_maxima_dias": max_days,
        "pedido_total_unidades_Q90": total_order,
        "inventario_objetivo_total_Q90": total_target
    })

coverage_sensitivity = pd.DataFrame(coverage_sensitivity_rows)
display(coverage_sensitivity)


,cobertura_maxima_dias,pedido_total_unidades_Q90,inventario_objetivo_total_Q90
0,7,11640,14932
1,8,13805,17128
2,9,15173,18515


## 12. Exportación de resultados

In [ ]:
recommendations.to_csv("recomendacion_abastecimiento_Q90.csv", index=False)
policy_comparison.to_csv("comparacion_politicas_Q85_Q90_Q95.csv", index=False)
policy_summary.to_csv("resumen_politicas_abastecimiento.csv", index=False)
coverage_sensitivity.to_csv("sensibilidad_cobertura_Q90.csv", index=False)
weekly_errors.to_csv("errores_semanales_backtesting_ets.csv", index=False)

print("Archivos exportados:")
for f in [
    "recomendacion_abastecimiento_Q90.csv",
    "comparacion_politicas_Q85_Q90_Q95.csv",
    "resumen_politicas_abastecimiento.csv",
    "sensibilidad_cobertura_Q90.csv",
    "errores_semanales_backtesting_ets.csv"
]:
    print("-", f)


Archivos exportados:
- recomendacion_abastecimiento_Q90.csv
- comparacion_politicas_Q85_Q90_Q95.csv
- resumen_politicas_abastecimiento.csv
- sensibilidad_cobertura_Q90.csv
- errores_semanales_backtesting_ets.csv


## 13. Limitaciones y validaciones pendientes con abastecimiento

1. ¿El stock reportado es inventario disponible o inventario físico?
2. ¿Existen pedidos en tránsito o recepciones programadas?
3. ¿Cuál es el lead time y la frecuencia de reposición?
4. ¿Las ventas históricas reflejan demanda real o quedan censuradas por quiebres?
5. ¿Qué productos tienen vencimiento, merma o restricciones de vida útil?
6. ¿Hay múltiplos de compra, mínimos de proveedor o restricciones de transporte?
7. ¿El día adicional de cobertura es operativo y comercialmente aceptable?
8. ¿Se dispone de capacidad real de almacenamiento por tienda y producto?

### Interpretación ejecutiva

La propuesta cubre la demanda pronosticada de siete días y añade una protección empírica moderada. Q90 se propone como política piloto; Q85 y Q95 permiten comparar el equilibrio entre disponibilidad e inventario. El límite de ocho días evita que la incertidumbre eleve automáticamente el objetivo a coberturas extremas.

La recomendación debe validarse con operación antes de emitir pedidos. En particular, al no contar con lead time, pedidos en tránsito ni capacidad real, este ejercicio representa una política de revisión semanal simplificada y no un sistema completo de reposición.
